# Dataset Cleaning

This notebook cleans the `dataset_no_tests` by removing:
1. Empty code snippets
2. Code containing `#include <criterion/criterion.h>` (remaining test files)
3. Entries with feedback "No functional error detected"

Output: `cleaned_dataset.csv` and `cleaned_dataset.jsonl`

## 1. Setup and Imports

In [1]:
import pandas as pd
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 2. Load Dataset

In [2]:
# Load the dataset without tests
dataset_path = Path('../data/raw_dataset_no_tests.jsonl')

data = []
with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

df = pd.DataFrame(data)

print(f"Loaded {len(df)} samples from dataset_no_tests")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDataset shape: {df.shape}")

Loaded 12232 samples from dataset_no_tests

Columns: ['code_id', 'author_id', 'code_snippet', 'generated_feedback']

Dataset shape: (12232, 4)


## 3. Initial Data Exploration

In [6]:
print("="*100)
print("INITIAL DATASET STATISTICS")
print("="*100)
print()

print(f"Total samples: {len(df)}")
print(f"\nNull values per column:")
print(df.isnull().sum())

# Check for empty code snippets
empty_code = df[df['code_snippet'].str.strip() == '']
print(f"\nEmpty code snippets: {len(empty_code)}")

# Check for criterion includes
criterion_code = df[df['code_snippet'].str.contains('#include <criterion/criterion.h>', regex=False, na=False)]
print(f"Code with criterion includes: {len(criterion_code)}")

# Check for "No functional error detected" feedback
no_error_feedback = df[df['generated_feedback'].str.contains('No functional error')]
print(f"Entries with 'No functional error: {len(no_error_feedback)}")

print(f"\nSample feedback values:")
print(df['generated_feedback'].value_counts().head(10))

INITIAL DATASET STATISTICS

Total samples: 12232

Null values per column:
code_id               0
author_id             0
code_snippet          0
generated_feedback    0
dtype: int64

Empty code snippets: 0
Code with criterion includes: 127
Entries with 'No functional error: 213

Sample feedback values:
generated_feedback
No functional errors.                                                                                                                       87
Consider the edge case where the input string is empty.                                                                                     37
When copying strings, ensure the destination has enough allocated space to hold the source string, including the null terminator.           24
Consider the base case when the input is exactly 1.                                                                                         19
Consider the behavior of the function when the source string is shorter than the specified number of cha

## 4. Data Cleaning

In [9]:
print("="*100)
print("CLEANING DATASET")
print("="*100)
print()

initial_count = len(df)
print(f"Starting with: {initial_count} samples")
print()

# Step 1: Remove empty code snippets
df_clean = df[df['code_snippet'].str.strip() != ''].copy()
removed_empty = initial_count - len(df_clean)
print(f"Step 1: Removed {removed_empty} empty code snippets")
print(f"        Remaining: {len(df_clean)} samples")
print()

# Step 2: Remove code containing criterion includes
before_criterion = len(df_clean)
df_clean = df_clean[~df_clean['code_snippet'].str.contains('#include <criterion/criterion.h>', regex=False, na=False)].copy()
removed_criterion = before_criterion - len(df_clean)
print(f"Step 2: Removed {removed_criterion} samples with criterion includes")
print(f"        Remaining: {len(df_clean)} samples")
print()

# Step 3: Remove entries with "No functional error detected"
before_no_error = len(df_clean)
df_clean = df_clean[~df_clean['generated_feedback'].str.contains('No functional error', regex=False, na=False)].copy()
removed_no_error = before_no_error - len(df_clean)
print(f"Step 3: Removed {removed_no_error} samples with 'No functional error'")
print(f"        Remaining: {len(df_clean)} samples")
print()

print("="*100)
print("CLEANING SUMMARY")
print("="*100)
total_removed = initial_count - len(df_clean)
print(f"Initial dataset: {initial_count} samples")
print(f"Final dataset:   {len(df_clean)} samples")
print(f"Total removed:   {total_removed} samples ({total_removed/initial_count*100:.2f}%)")
print()
print(f"Breakdown:")
print(f"  - Empty code snippets:          {removed_empty}")
print(f"  - Criterion test includes:      {removed_criterion}")
print(f"  - No functional error: {removed_no_error}")

CLEANING DATASET

Starting with: 12232 samples

Step 1: Removed 0 empty code snippets
        Remaining: 12232 samples

Step 2: Removed 127 samples with criterion includes
        Remaining: 12105 samples

Step 3: Removed 207 samples with 'No functional error'
        Remaining: 11898 samples

CLEANING SUMMARY
Initial dataset: 12232 samples
Final dataset:   11898 samples
Total removed:   334 samples (2.73%)

Breakdown:
  - Empty code snippets:          0
  - Criterion test includes:      127
  - No functional error: 207


## 5. Validate Cleaned Data

In [10]:
print("="*100)
print("CLEANED DATASET VALIDATION")
print("="*100)
print()

# Verify no empty code
empty_check = df_clean[df_clean['code_snippet'].str.strip() == '']
print(f"Empty code snippets: {len(empty_check)} ✓" if len(empty_check) == 0 else f"Empty code snippets: {len(empty_check)} ✗")

# Verify no criterion includes
criterion_check = df_clean[df_clean['code_snippet'].str.contains('#include <criterion/criterion.h>', regex=False, na=False)]
print(f"Criterion includes: {len(criterion_check)} ✓" if len(criterion_check) == 0 else f"Criterion includes: {len(criterion_check)} ✗")

# Verify no "No functional error detected"
no_error_check = df_clean[df_clean['generated_feedback'].str.contains('No functional error detected')]
print(f"'No functional error detected': {len(no_error_check)} ✓" if len(no_error_check) == 0 else f"'No functional error detected': {len(no_error_check)} ✗")

print()
print("Cleaned dataset statistics:")
print(f"  Total samples: {len(df_clean)}")
print(f"  Null values: {df_clean.isnull().sum().sum()}")
print(f"  Unique code_ids: {df_clean['code_id'].nunique()}")
print(f"  Unique authors: {df_clean['author_id'].nunique()}")

print("\nTop 10 feedback types in cleaned dataset:")
print(df_clean['generated_feedback'].value_counts().head(10))

CLEANED DATASET VALIDATION

Empty code snippets: 0 ✓
Criterion includes: 0 ✓
'No functional error detected': 0 ✓

Cleaned dataset statistics:
  Total samples: 11898
  Null values: 0
  Unique code_ids: 11898
  Unique authors: 1320

Top 10 feedback types in cleaned dataset:
generated_feedback
Consider the edge case where the input string is empty.                                                                                     37
When copying strings, ensure the destination has enough allocated space to hold the source string, including the null terminator.           24
Consider the behavior of the function when the source string is shorter than the specified number of characters.                            19
Consider the base case when the input is exactly 1.                                                                                         19
When copying strings, ensure the destination has enough allocated space to accommodate the source string, including the null terminator.

## 6. Sample Cleaned Data

In [ ]:
print("="*100)
print("SAMPLE CLEANED DATA")
print("="*100)
print()

# Show 3 random samples
samples = df_clean.sample(3)

for idx, row in samples.iterrows():
    print(f"Sample {idx + 1}:")
    print(f"Code ID: {row['code_id']}")
    print(f"Author ID: {row['author_id']}")
    print(f"\nCode snippet (first 200 chars):")
    print(row['code_snippet'][:200] + ("..." if len(row['code_snippet']) > 200 else ""))
    print(f"\nFeedback:")
    print(row['generated_feedback'][:200] + ("..." if len(row['generated_feedback']) > 200 else ""))
    print("\n" + "-"*100 + "\n")

## 7. Save Cleaned Dataset

In [11]:
# Save as CSV
csv_output_path = Path('../data/cleaned_dataset.csv')
df_clean.to_csv(csv_output_path, index=False)
print(f"✓ Cleaned dataset saved to CSV: {csv_output_path}")
print(f"  Size: {csv_output_path.stat().st_size / 1024 / 1024:.2f} MB")

# Save as JSONL
jsonl_output_path = Path('../data/cleaned_dataset.jsonl')
with open(jsonl_output_path, 'w', encoding='utf-8') as f:
    for _, row in df_clean.iterrows():
        json.dump(row.to_dict(), f, ensure_ascii=False)
        f.write('\n')
print(f"✓ Cleaned dataset saved to JSONL: {jsonl_output_path}")
print(f"  Size: {jsonl_output_path.stat().st_size / 1024 / 1024:.2f} MB")

print(f"\n✓ Cleaning complete!")
print(f"  Final dataset: {len(df_clean)} samples")

✓ Cleaned dataset saved to CSV: ../data/cleaned_dataset.csv
  Size: 6.14 MB
✓ Cleaned dataset saved to JSONL: ../data/cleaned_dataset.jsonl
  Size: 7.15 MB

✓ Cleaning complete!
  Final dataset: 11898 samples


## 8. Dataset Statistics

In [12]:
print("="*100)
print("FINAL DATASET STATISTICS")
print("="*100)
print()

print(f"Dataset: cleaned_dataset")
print(f"Total samples: {len(df_clean):,}")
print()

print("Code snippet statistics:")
code_lengths = df_clean['code_snippet'].str.len()
print(f"  Average length: {code_lengths.mean():.0f} characters")
print(f"  Median length: {code_lengths.median():.0f} characters")
print(f"  Min length: {code_lengths.min()} characters")
print(f"  Max length: {code_lengths.max()} characters")
print()

print("Feedback statistics:")
feedback_lengths = df_clean['generated_feedback'].str.len()
print(f"  Average length: {feedback_lengths.mean():.0f} characters")
print(f"  Median length: {feedback_lengths.median():.0f} characters")
print(f"  Unique feedback messages: {df_clean['generated_feedback'].nunique():,}")
print()

print("Author statistics:")
print(f"  Unique authors: {df_clean['author_id'].nunique():,}")
print(f"  Average samples per author: {len(df_clean) / df_clean['author_id'].nunique():.1f}")
print()

print("="*100)

FINAL DATASET STATISTICS

Dataset: cleaned_dataset
Total samples: 11,898

Code snippet statistics:
  Average length: 289 characters
  Median length: 245 characters
  Min length: 32 characters
  Max length: 2932 characters

Feedback statistics:
  Average length: 173 characters
  Median length: 154 characters
  Unique feedback messages: 11,152

Author statistics:
  Unique authors: 1,320
  Average samples per author: 9.0

